# Análise da Concentração de Energia nos Primeiros Anéis de Cada Subdetector (Anéis de Pico)

Nos perfis médios de energia normalizada ($L_1$) dos anéis do calorímetro (`PreSampler`, `EM1`, `EM2`, `EM3` e `TileCal`), observa-se que os **3 primeiros anéis** de cada subdetector concentram um pico acentuado da energia depositada.

Este notebook implementa uma análise em streaming (baixo consumo de memória) comparando **Sinal (Fótons)** e **Fundo (Jatos Hadrônicos)** para os modelos de **100 anéis** e **50 anéis**, utilizando as classes OOP `RingPeakEnergyAnalyzer` e `RingPeakEnergyPlotter`:

1. **Fração Intra-Detector ($\%_{\text{intra}}$)**: porcentagem que a soma dos primeiros $k=3$ anéis de um subdetector $d$ representa dentro da energia total daquele mesmo subdetector:
   $$\text{Pct}_{\text{intra}}(d) = \frac{\sum_{i=1}^{3} \bar{e}_{d,i}}{\sum_{i \in d} \bar{e}_{d,i}} \times 100\%$$
2. **Fração Global no Calorímetro ($\%_{\text{global}}$)**: porcentagem que a soma dos primeiros $k=3$ anéis de um subdetector $d$ representa em relação à energia total de todos os anéis do calorímetro:
   $$\text{Pct}_{\text{global}}(d) = \frac{\sum_{i=1}^{3} \bar{e}_{d,i}}{\sum_{j \in \text{todos}} \bar{e}_{j}} \times 100\%$$
3. **Partição Padrão de 5 Subdetectores (100 Anéis)**:
   - `PreSampler`: anéis `[0..7]` (8 anéis)
   - `EM1`: anéis `[8..71]` (64 anéis)
   - `EM2`: anéis `[72..79]` (8 anéis)
   - `EM3`: anéis `[80..87]` (8 anéis)
   - `TileCal`: anéis `[88..99]` (12 anéis)

In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplhep as mh
from IPython.display import display

mh.style.use("ATLAS")

from src.core.Datasets.EgammaNpzDataset import EgammaNpzDataset
from src.core.Plotting.Localization import PlotTextKey, PlotTranslator
from src.core.Plotting.ProfilePlotter import (
    RingPeakEnergyPlotter,
    StreamingRingProfileAccumulator,
)
from src.core.Validation.RingPeakEnergyAnalyzer import RingPeakEnergyAnalyzer
from src.utils import get_et_eta

print(f"Project root: {PROJECT_ROOT}")

### 1. Configuração, Caminhos e Parâmetros da Análise
Defina os caminhos de configuração, o número de anéis de pico (`TOP_K_RINGS = 3`), o idioma dos gráficos (`LANGUAGE`) e as opções de execução por região.

In [ ]:
CONFIG_PATH_100 = PROJECT_ROOT / "config/NeuralRinger/ModelV10_HighBatch_100Rings.yaml"
CONFIG_PATH_50 = PROJECT_ROOT / "config/NeuralRinger/ModelV10_HighBatch_50Rings.yaml"

with open(CONFIG_PATH_100, "r") as f:
    config_dict_100 = yaml.safe_load(f)

DATA_PATH = config_dict_100["dataset"]["parameters"]["drive_path"]
if not os.path.exists(DATA_PATH):
    local_data_path = PROJECT_ROOT / "data/consolidated"
    if local_data_path.exists():
        DATA_PATH = str(local_data_path)

print(f"Dataset path: {DATA_PATH}")

# Controle de execução
READ_ALL_DATASETS = True      # False processa apenas a primeira região (teste rápido)
SHOW_REGIONAL_PLOTS = False   # True exibe todos os gráficos regionais inline; False exibe apenas a 1ª região
TOP_K_RINGS = 3               # Primeiros K anéis de cada subdetector
PLOT_FORMAT = "pdf"           # Formato dos gráficos salvos ('pdf' ou 'png')

# Idioma dos gráficos ('pt' ou 'en')
LANGUAGE = PlotTranslator.validate_language("pt")
print(f"Plot language: {LANGUAGE} | TOP_K_RINGS={TOP_K_RINGS}")

# Diretórios de saída
OUTPUT_DIR = PROJECT_ROOT / "Plots/ring_peak_energy_analysis"
TABLES_DIR = OUTPUT_DIR / "Tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")
print(f"Tables dir: {TABLES_DIR}")

### 2. Instanciação dos Datasets e Mapeamento dos Anéis por Subdetector
Instanciamos `dataset_100` (`percentage=1.0`) e `dataset_50` (`percentage=0.5`) e inspecionamos exatamente quais anéis globais e posições locais correspondem aos 3 primeiros anéis de cada subdetector.

In [ ]:
startswith = config_dict_100["dataset"]["parameters"].get("startswith", "consolidated.et")
endswith = config_dict_100["dataset"]["parameters"].get("endswith", ".npz")

dataset_100 = EgammaNpzDataset(
    drive_path=DATA_PATH,
    startswith=startswith,
    endswith=endswith,
    percentage=1.0,
)
dataset_50 = EgammaNpzDataset(
    drive_path=DATA_PATH,
    startswith=startswith,
    endswith=endswith,
    percentage=0.5,
)

feature_names = dataset_100.feature_names
dataset_100._set_ring_column_indices(feature_names)
dataset_50._set_ring_column_indices(feature_names)

analyzer = RingPeakEnergyAnalyzer(top_k=TOP_K_RINGS)
peak_plotter = RingPeakEnergyPlotter(language=LANGUAGE, top_k=TOP_K_RINGS)

print(f"Arquivos de região descobertos: {len(dataset_100)}")
print(f"Modelo 100 Anéis: {dataset_100.percentage_dim} anéis ativos")
print(f"Modelo 50 Anéis:  {dataset_50.percentage_dim} anéis ativos")

print("\nMapeamento de Subdetectores - Modelo 100 Anéis:")
display(analyzer.describe_subdetector_mapping(dataset_100.percentage_dim, dataset_100.indexes))

print("\nMapeamento de Subdetectores - Modelo 50 Anéis:")
display(analyzer.describe_subdetector_mapping(dataset_50.percentage_dim, dataset_50.indexes))

### 3. Processamento em Streaming por Região ($E_T \times \eta$)
Lemos um arquivo `.npz` por vez em ordem crescente de $(E_T, \eta)$, acumulamos as estatísticas globais com memória $O(1)$ via `StreamingRingProfileAccumulator`, calculamos as frações intra-detector e globais por região e liberamos a memória imediatamente após cada região.

In [ ]:
sorted_indices = sorted(
    range(len(dataset_100)),
    key=lambda idx: get_et_eta(dataset_100.file_paths[idx]),
)
files_to_process = sorted_indices if READ_ALL_DATASETS else sorted_indices[:1]
print(f"Processando {len(files_to_process)} região(ões) (READ_ALL_DATASETS={READ_ALL_DATASETS})\n")

acc_100 = StreamingRingProfileAccumulator(
    n_rings=dataset_100.percentage_dim,
    ring_indexes=np.asarray(dataset_100.indexes),
)
acc_50 = StreamingRingProfileAccumulator(
    n_rings=dataset_50.percentage_dim,
    ring_indexes=np.asarray(dataset_50.indexes),
)

regional_dfs_100 = []
regional_dfs_50 = []
regional_overview_rows = []

sig_label = PlotTranslator.translate(PlotTextKey.PROFILE_SIGNAL_LABEL, language=LANGUAGE)
bg_label = PlotTranslator.translate(PlotTextKey.PROFILE_BACKGROUND_LABEL, language=LANGUAGE)

for loop_idx, idx in enumerate(files_to_process):
    data, labels, file_path = dataset_100[idx]
    et, eta = get_et_eta(file_path)
    region_key = f"et{int(et)}_eta{int(eta)}"

    rings_100 = data[:, dataset_100.ring_column_indices]
    rings_50 = data[:, dataset_50.ring_column_indices]

    acc_100.update(rings_100, labels)
    acc_50.update(rings_50, labels)

    reg_acc_100 = StreamingRingProfileAccumulator(
        n_rings=dataset_100.percentage_dim,
        ring_indexes=np.asarray(dataset_100.indexes),
    )
    reg_acc_100.update(rings_100, labels)
    mean_sig_100, std_sig_100, mean_bg_100, std_bg_100 = reg_acc_100.compute_stats()

    reg_acc_50 = StreamingRingProfileAccumulator(
        n_rings=dataset_50.percentage_dim,
        ring_indexes=np.asarray(dataset_50.indexes),
    )
    reg_acc_50.update(rings_50, labels)
    mean_sig_50, std_sig_50, mean_bg_50, std_bg_50 = reg_acc_50.compute_stats()

    df_reg_100 = analyzer.analyze_signal_vs_background(
        mean_signal=mean_sig_100,
        mean_bkg=mean_bg_100,
        ring_indexes=np.asarray(dataset_100.indexes),
        region=region_key,
        std_signal=std_sig_100,
        std_bkg=std_bg_100,
    )
    df_reg_50 = analyzer.analyze_signal_vs_background(
        mean_signal=mean_sig_50,
        mean_bkg=mean_bg_50,
        ring_indexes=np.asarray(dataset_50.indexes),
        region=region_key,
        std_signal=std_sig_50,
        std_bkg=std_bg_50,
    )

    regional_dfs_100.append(df_reg_100)
    regional_dfs_50.append(df_reg_50)

    conc_100 = analyzer.compute_total_peak_concentration(df_reg_100)
    conc_50 = analyzer.compute_total_peak_concentration(df_reg_50)
    regional_overview_rows.append(
        {
            "region": region_key,
            "et": int(et),
            "eta": int(eta),
            "n_signal": reg_acc_100.n_signal,
            "n_bg": reg_acc_100.n_bg,
            "sig_15peaks_100rings_global_pct": conc_100["signal_total_peak_global_pct"],
            "bkg_15peaks_100rings_global_pct": conc_100["background_total_peak_global_pct"],
            "sig_15peaks_50rings_global_pct": conc_50["signal_total_peak_global_pct"],
            "bkg_15peaks_50rings_global_pct": conc_50["background_total_peak_global_pct"],
        }
    )

    show_inline = SHOW_REGIONAL_PLOTS or (loop_idx == 0)
    print(
        f"[{loop_idx + 1}/{len(files_to_process)}] {region_key} | "
        f"{sig_label}: {reg_acc_100.n_signal:,} | {bg_label}: {reg_acc_100.n_bg:,} | "
        f"Top-15/100 anéis: Sinal={conc_100['signal_total_peak_global_pct']:.2f}%, "
        f"Fundo={conc_100['background_total_peak_global_pct']:.2f}%"
    )

    peak_plotter.plot_highlighted_peak_profile(
        mean_signal=mean_sig_100,
        std_signal=std_sig_100,
        mean_bkg=mean_bg_100,
        std_bkg=std_bg_100,
        output_dir=OUTPUT_DIR,
        filename=f"{region_key}_peak_profile_100Rings.{PLOT_FORMAT}",
        ring_indexes=np.asarray(dataset_100.indexes),
        scope_label=region_key,
        language=LANGUAGE,
        show=show_inline,
        subfolder="RegionalProfiles",
    )
    peak_plotter.plot_subdetector_peak_bars(
        comparison_df=df_reg_100,
        output_dir=OUTPUT_DIR,
        filename=f"{region_key}_peak_bars_100Rings.{PLOT_FORMAT}",
        scope_label=region_key,
        language=LANGUAGE,
        show=show_inline,
        subfolder="RegionalBars",
    )

    del data, labels, rings_100, rings_50, reg_acc_100, reg_acc_50
    gc.collect()

all_regional_100 = pd.concat(regional_dfs_100, ignore_index=True)
all_regional_50 = pd.concat(regional_dfs_50, ignore_index=True)
df_regional_overview = pd.DataFrame(regional_overview_rows)

all_regional_100.to_csv(TABLES_DIR / "regional_peak_energy_100rings.csv", index=False)
all_regional_50.to_csv(TABLES_DIR / "regional_peak_energy_50rings.csv", index=False)
df_regional_overview.to_csv(TABLES_DIR / "regional_total_peak_concentration.csv", index=False)

print("\nResumo Regional Consolidado (15 Anéis de Pico / Total do Calorímetro):")
display(df_regional_overview.round(2))

### 4. Análise Global (Todas as Regiões Consolidadas) — 100 Anéis e 50 Anéis
Calculamos as porcentagens sobre o perfil médio global consolidado de todo o conjunto de dados (`acc_100` e `acc_50`), exibindo:
- A contribuição individual dos anéis 1, 2 e 3 de cada subdetector;
- A **% Intra-Detector** (quanto os 3 primeiros anéis representam dentro do próprio subdetector);
- A **% Global no Calorímetro** (quanto os 3 primeiros anéis representam da energia total de todos os anéis);
- A **Concentração Acumulada nos 15 Anéis de Pico** (5 subdetectores $\times$ 3 primeiros anéis).

In [ ]:
def format_global_table(df_comp: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "Subdetector": df_comp["subdetector"],
            "Anéis (Subdet.)": df_comp["n_rings_subdetector"],
            "Anéis Pico (Globais)": df_comp["peak_global_rings"].astype(str),
            "Sinal Intra-Det (%)": df_comp["signal_intra_detector_pct"].round(2),
            "Fundo Intra-Det (%)": df_comp["background_intra_detector_pct"].round(2),
            "Delta Intra (pp)": df_comp["delta_intra_detector_pct"].round(2),
            "Sinal Global (%)": df_comp["signal_global_calorimeter_pct"].round(2),
            "Fundo Global (%)": df_comp["background_global_calorimeter_pct"].round(2),
            "Delta Global (pp)": df_comp["delta_global_calorimeter_pct"].round(2),
            "Fração Subdet. Sinal (%)": df_comp["signal_subdetector_share_pct"].round(2),
            "Fração Subdet. Fundo (%)": df_comp["background_subdetector_share_pct"].round(2),
        }
    )

# --- 100 Anéis ---
g_sig_mean_100, g_sig_std_100, g_bg_mean_100, g_bg_std_100 = acc_100.compute_stats()
global_df_100 = analyzer.analyze_signal_vs_background(
    mean_signal=g_sig_mean_100,
    mean_bkg=g_bg_mean_100,
    ring_indexes=np.asarray(dataset_100.indexes),
    region=f"Global ({len(files_to_process)} regiões)",
    std_signal=g_sig_std_100,
    std_bkg=g_bg_std_100,
)
conc_global_100 = analyzer.compute_total_peak_concentration(global_df_100)

# --- 50 Anéis ---
g_sig_mean_50, g_sig_std_50, g_bg_mean_50, g_bg_std_50 = acc_50.compute_stats()
global_df_50 = analyzer.analyze_signal_vs_background(
    mean_signal=g_sig_mean_50,
    mean_bkg=g_bg_mean_50,
    ring_indexes=np.asarray(dataset_50.indexes),
    region=f"Global ({len(files_to_process)} regiões)",
    std_signal=g_sig_std_50,
    std_bkg=g_bg_std_50,
)
conc_global_50 = analyzer.compute_total_peak_concentration(global_df_50)

print("=" * 80)
print(f"ANÁLISE GLOBAL - MODELO 100 ANÉIS (Sinal N={acc_100.n_signal:,}, Fundo N={acc_100.n_bg:,})")
print("=" * 80)
print(
    f"Total nos {int(conc_global_100['total_peak_rings'])}/{int(conc_global_100['total_rings'])} anéis de pico: "
    f"Sinal = {conc_global_100['signal_total_peak_global_pct']:.2f}% | "
    f"Fundo = {conc_global_100['background_total_peak_global_pct']:.2f}% | "
    f"Delta = {conc_global_100['delta_total_peak_global_pct']:+.2f} pp"
)
display(format_global_table(global_df_100))

print("\n" + "=" * 80)
print(f"ANÁLISE GLOBAL - MODELO 50 ANÉIS (Sinal N={acc_50.n_signal:,}, Fundo N={acc_50.n_bg:,})")
print("=" * 80)
print(
    f"Total nos {int(conc_global_50['total_peak_rings'])}/{int(conc_global_50['total_rings'])} anéis de pico: "
    f"Sinal = {conc_global_50['signal_total_peak_global_pct']:.2f}% | "
    f"Fundo = {conc_global_50['background_total_peak_global_pct']:.2f}% | "
    f"Delta = {conc_global_50['delta_total_peak_global_pct']:+.2f} pp"
)
display(format_global_table(global_df_50))

# Gráficos globais (100 e 50 Anéis)
scope_text = f"Global ({len(files_to_process)} regiões)" if LANGUAGE == "pt" else f"Global ({len(files_to_process)} regions)"

peak_plotter.plot_highlighted_peak_profile(
    mean_signal=g_sig_mean_100,
    std_signal=g_sig_std_100,
    mean_bkg=g_bg_mean_100,
    std_bkg=g_bg_std_100,
    output_dir=OUTPUT_DIR,
    filename=f"global_peak_profile_100Rings.{PLOT_FORMAT}",
    ring_indexes=np.asarray(dataset_100.indexes),
    scope_label=scope_text,
    language=LANGUAGE,
    show=True,
    subfolder="Global",
)
peak_plotter.plot_subdetector_peak_bars(
    comparison_df=global_df_100,
    output_dir=OUTPUT_DIR,
    filename=f"global_peak_bars_100Rings.{PLOT_FORMAT}",
    scope_label=scope_text,
    language=LANGUAGE,
    show=True,
    subfolder="Global",
)

peak_plotter.plot_highlighted_peak_profile(
    mean_signal=g_sig_mean_50,
    std_signal=g_sig_std_50,
    mean_bkg=g_bg_mean_50,
    std_bkg=g_bg_std_50,
    output_dir=OUTPUT_DIR,
    filename=f"global_peak_profile_50Rings.{PLOT_FORMAT}",
    ring_indexes=np.asarray(dataset_50.indexes),
    scope_label=scope_text,
    language=LANGUAGE,
    show=True,
    subfolder="Global",
)
peak_plotter.plot_subdetector_peak_bars(
    comparison_df=global_df_50,
    output_dir=OUTPUT_DIR,
    filename=f"global_peak_bars_50Rings.{PLOT_FORMAT}",
    scope_label=scope_text,
    language=LANGUAGE,
    show=True,
    subfolder="Global",
)

# Exportação das tabelas globais
global_df_100.to_csv(TABLES_DIR / "global_peak_energy_100rings.csv", index=False)
global_df_50.to_csv(TABLES_DIR / "global_peak_energy_50rings.csv", index=False)
summary_conc_df = pd.DataFrame(
    [
        {"model": "100 Rings", **conc_global_100},
        {"model": "50 Rings", **conc_global_50},
    ]
)
summary_conc_df.to_csv(TABLES_DIR / "global_total_peak_concentration_summary.csv", index=False)

### 5. Decomposição Individual dos Anéis 1, 2 e 3 (Modelo Global de 100 Anéis)
Inspeção da contribuição relativa de cada um dos três primeiros anéis dentro do seu subdetector.

In [ ]:
ring_breakdown_df = pd.DataFrame(
    {
        "Subdetector": global_df_100["subdetector"],
        "Sinal Anel 1 (% intra)": global_df_100["signal_ring_1_intra_pct"].round(2),
        "Sinal Anel 2 (% intra)": global_df_100["signal_ring_2_intra_pct"].round(2),
        "Sinal Anel 3 (% intra)": global_df_100["signal_ring_3_intra_pct"].round(2),
        "Sinal Soma Top-3 (% intra)": global_df_100["signal_intra_detector_pct"].round(2),
        "Fundo Anel 1 (% intra)": global_df_100["background_ring_1_intra_pct"].round(2),
        "Fundo Anel 2 (% intra)": global_df_100["background_ring_2_intra_pct"].round(2),
        "Fundo Anel 3 (% intra)": global_df_100["background_ring_3_intra_pct"].round(2),
        "Fundo Soma Top-3 (% intra)": global_df_100["background_intra_detector_pct"].round(2),
    }
)
display(ring_breakdown_df)
ring_breakdown_df.to_csv(TABLES_DIR / "global_individual_ring_breakdown_100rings.csv", index=False)

### 6. Variação Cinemática nas Regiões ($E_T \times \eta$)
Analisamos a estabilidade (média $\pm$ desvio padrão, mínimo e máximo) das frações de pico ao longo das regiões cinemáticas e geramos os mapas de calor 2D (`Região × Subdetector`) para 100 Anéis e 50 Anéis.

In [ ]:
variation_100 = analyzer.summarize_across_regions(all_regional_100)
variation_50 = analyzer.summarize_across_regions(all_regional_50)

variation_100.to_csv(TABLES_DIR / "regional_variation_summary_100rings.csv", index=False)
variation_50.to_csv(TABLES_DIR / "regional_variation_summary_50rings.csv", index=False)

print("Variação Entre Regiões Cinemáticas - 100 Anéis:")
display(variation_100.round(2))

print("\nVariação Entre Regiões Cinemáticas - 50 Anéis:")
display(variation_50.round(2))

for metric_name in ("intra_detector_pct", "global_calorimeter_pct"):
    peak_plotter.plot_regional_peak_heatmap(
        regional_df=all_regional_100,
        output_dir=OUTPUT_DIR,
        filename=f"heatmap_regional_{metric_name}_100Rings.{PLOT_FORMAT}",
        metric=metric_name,
        language=LANGUAGE,
        show=True,
        subfolder="RegionalHeatmaps",
    )
    peak_plotter.plot_regional_peak_heatmap(
        regional_df=all_regional_50,
        output_dir=OUTPUT_DIR,
        filename=f"heatmap_regional_{metric_name}_50Rings.{PLOT_FORMAT}",
        metric=metric_name,
        language=LANGUAGE,
        show=True,
        subfolder="RegionalHeatmaps",
    )

### 7. Análise da Variância Média por Subdetector e Calorímetro Hadrônico (`TileCal` vs Camadas EM)
Avaliamos se a região do calorímetro hadrônico (`TileCal`) apresenta maior dispersão energética em comparação às camadas eletromagnéticas (`PreSampler`, `EM1`, `EM2`, `EM3`), calculando tanto sobre **todos os anéis do subdetector** quanto especificamente sobre os **3 primeiros anéis (pico)**:
- **Variância média absoluta por anel** ($\overline{\sigma^2}$): dispersão na escala bruta da fração de energia $L_1$, onde camadas de maior energia depositada (`EM1`, `EM2`) tendem a concentrar a variância absoluta, mas jatos hadrônicos (Fundo) aumentam fortemente a variância em `TileCal` em comparação a fótons (Sinal);
- **Coeficiente de Variação relativo** ($\mathrm{CV} = \overline{\sigma} / |\overline{\mu}|$): dispersão normalizada pela magnitude média da energia na camada, revelando a instabilidade relativa do depósito hadrônico em `TileCal`;
- **Fração da Variância Total** ($\mathrm{Var}_d / \mathrm{Var}_{\text{total}} \times 100\%$): contribuição de cada subdetector para a variância acumulada no calorímetro.

In [ ]:
def format_variance_table(df_comp: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "Subdetector": df_comp["subdetector"],
            "Sinal Variância Média Anel (σ²)": df_comp["signal_mean_ring_variance"],
            "Fundo Variância Média Anel (σ²)": df_comp["background_mean_ring_variance"],
            "Sinal Variância Top-3 (σ²)": df_comp["signal_peak_mean_ring_variance"],
            "Fundo Variância Top-3 (σ²)": df_comp["background_peak_mean_ring_variance"],
            "Razão Variância (Fundo/Sinal)": df_comp["ratio_bkg_to_sig_mean_variance"].round(2),
            "Sinal CV (σ/|μ|)": df_comp["signal_subdetector_cv"].round(3),
            "Fundo CV (σ/|μ|)": df_comp["background_subdetector_cv"].round(3),
            "Sinal CV Top-3": df_comp["signal_peak_cv"].round(3),
            "Fundo CV Top-3": df_comp["background_peak_cv"].round(3),
            "% Variância Total Sinal": df_comp["signal_variance_share_pct"].round(2),
            "% Variância Total Fundo": df_comp["background_variance_share_pct"].round(2),
        }
    )

var_table_100 = format_variance_table(global_df_100)
var_table_50 = format_variance_table(global_df_50)

print("=" * 80)
print("VARIÂNCIA MÉDIA E DISPERSÃO RELATIVA POR SUBDETECTOR - MODELO 100 ANÉIS")
print("=" * 80)
display(var_table_100)

print("\n" + "=" * 80)
print("VARIÂNCIA MÉDIA E DISPERSÃO RELATIVA POR SUBDETECTOR - MODELO 50 ANÉIS")
print("=" * 80)
display(var_table_50)

had_vs_em_100 = analyzer.compare_hadronic_vs_em_variance(global_df_100)
had_vs_em_50 = analyzer.compare_hadronic_vs_em_variance(global_df_50)

had_vs_em_df = pd.DataFrame(
    [
        {"model": "100 Rings", **had_vs_em_100},
        {"model": "50 Rings", **had_vs_em_50},
    ]
)

print("\nDiagnóstico Comparativo: Calorímetro Hadrônico (TileCal) vs Camadas EM:")
for model_name, summary in (("100 Anéis", had_vs_em_100), ("50 Anéis", had_vs_em_50)):
    print(f"\n[{model_name}]")
    print(
        f"  Variância absoluta (Sinal) -> TileCal: {summary['signal_hadronic_mean_variance']:.3e} | "
        f"Média EM: {summary['signal_em_mean_variance']:.3e} "
        f"(Razão TileCal/EM: {summary['signal_hadronic_to_em_variance_ratio']:.3f}x) | "
        f"Maior var. abs.: {summary['signal_highest_abs_variance_subdetector']}"
    )
    print(
        f"  Variância absoluta (Fundo) -> TileCal: {summary['background_hadronic_mean_variance']:.3e} | "
        f"Média EM: {summary['background_em_mean_variance']:.3e} "
        f"(Razão TileCal/EM: {summary['background_hadronic_to_em_variance_ratio']:.3f}x) | "
        f"Maior var. abs.: {summary['background_highest_abs_variance_subdetector']}"
    )
    print(
        f"  Dispersão relativa CV (Sinal) -> TileCal: {summary['signal_hadronic_cv']:.3f} | "
        f"Média EM: {summary['signal_em_mean_cv']:.3f} "
        f"(Razão TileCal/EM: {summary['signal_hadronic_to_em_cv_ratio']:.3f}x) | "
        f"Maior CV: {summary['signal_highest_cv_subdetector']}"
    )
    print(
        f"  Dispersão relativa CV (Fundo) -> TileCal: {summary['background_hadronic_cv']:.3f} | "
        f"Média EM: {summary['background_em_mean_cv']:.3f} "
        f"(Razão TileCal/EM: {summary['background_hadronic_to_em_cv_ratio']:.3f}x) | "
        f"Maior CV: {summary['background_highest_cv_subdetector']}"
    )

peak_plotter.plot_subdetector_variance_bars(
    comparison_df=global_df_100,
    output_dir=OUTPUT_DIR,
    filename=f"global_subdetector_variance_100Rings.{PLOT_FORMAT}",
    scope_label=scope_text,
    language=LANGUAGE,
    show=True,
    subfolder="Global",
)
peak_plotter.plot_subdetector_variance_bars(
    comparison_df=global_df_50,
    output_dir=OUTPUT_DIR,
    filename=f"global_subdetector_variance_50Rings.{PLOT_FORMAT}",
    scope_label=scope_text,
    language=LANGUAGE,
    show=True,
    subfolder="Global",
)

for metric_name in ("mean_ring_variance", "subdetector_cv"):
    peak_plotter.plot_regional_peak_heatmap(
        regional_df=all_regional_100,
        output_dir=OUTPUT_DIR,
        filename=f"heatmap_regional_{metric_name}_100Rings.{PLOT_FORMAT}",
        metric=metric_name,
        language=LANGUAGE,
        show=True,
        subfolder="RegionalHeatmaps",
    )
    peak_plotter.plot_regional_peak_heatmap(
        regional_df=all_regional_50,
        output_dir=OUTPUT_DIR,
        filename=f"heatmap_regional_{metric_name}_50Rings.{PLOT_FORMAT}",
        metric=metric_name,
        language=LANGUAGE,
        show=True,
        subfolder="RegionalHeatmaps",
    )

var_table_100.to_csv(TABLES_DIR / "global_subdetector_variance_100rings.csv", index=False)
var_table_50.to_csv(TABLES_DIR / "global_subdetector_variance_50rings.csv", index=False)
had_vs_em_df.to_csv(TABLES_DIR / "hadronic_vs_em_variance_comparison.csv", index=False)

### 8. Histograma 2D: Energia Transversa (`trig_L2_calo_et`) vs Região do Detector (Por Arquivo de Região)
Para cada arquivo individual de região ($E_T \times \eta$), geramos um histograma 2D comparando **Sinal (Fótons)** e **Fundo (Jatos Hadrônicos)** lado a lado, relacionando a energia transversa `trig_L2_calo_et` (em GeV) com cada subdetector do calorímetro (`PreSampler`, `EM1`, `EM2`, `EM3`, `TileCal`), ponderado pela energia normalizada ($L_1$) depositada nos anéis de cada camada.

> Esta análise é executada exclusivamente por arquivo de região (sem consolidação global).

In [ ]:
ET_FEATURE_NAME = "trig_L2_calo_et"
N_ET_BINS = 30

feature_names_str = [str(name) for name in feature_names]
if ET_FEATURE_NAME not in feature_names_str:
    raise KeyError(f"Feature '{ET_FEATURE_NAME}' not found in dataset feature_names.")
et_col_idx = feature_names_str.index(ET_FEATURE_NAME)

print(f"Gerando histogramas 2D ({ET_FEATURE_NAME} [GeV] vs Subdetector) para {len(files_to_process)} região(ões)...\n")

for loop_idx, idx in enumerate(files_to_process):
    data, labels, file_path = dataset_100[idx]
    et, eta = get_et_eta(file_path)
    region_key = f"et{int(et)}_eta{int(eta)}"

    et_gev = data[:, et_col_idx].astype(np.float64) / 1000.0
    rings_100 = data[:, dataset_100.ring_column_indices]

    hist2d_data = analyzer.compute_subdetector_et_histogram2d(
        rings_data=rings_100,
        transverse_energy_gev=et_gev,
        labels=labels,
        ring_indexes=np.asarray(dataset_100.indexes),
        n_et_bins=N_ET_BINS,
    )

    show_inline = SHOW_REGIONAL_PLOTS or (loop_idx == 0)
    saved_path = peak_plotter.plot_subdetector_et_hist2d(
        hist2d_data=hist2d_data,
        output_dir=OUTPUT_DIR,
        filename=f"{region_key}_et_vs_subdetector_hist2d_100Rings.{PLOT_FORMAT}",
        scope_label=region_key,
        et_label=ET_FEATURE_NAME,
        language=LANGUAGE,
        show=show_inline,
        subfolder="RegionalEtHist2D",
    )
    print(
        f"[{loop_idx + 1}/{len(files_to_process)}] {region_key} | "
        f"{sig_label}: {hist2d_data['n_signal']:,} | {bg_label}: {hist2d_data['n_background']:,} | "
        f"Salvo em: {Path(saved_path).name}"
    )

    del data, labels, rings_100, et_gev, hist2d_data
    gc.collect()